<h1 style="color: rgba(237, 153, 29, 1);">Manipulação de Arquivos Binários e Argumentos de Linha de Comando</h1>

<small>
<p><strong>IMPORTANTE</strong>: O comando '%%file' é usado no Python para criar arquivos .c no diretório onde este notebook está salvo. Os arquivos criados são nomeados conforme o identificador fornecido após o comando '%%file'.</p>
</small>

<h3>Arquivos Binários</h3>

<p>Ao manipular um arquivo no modo <strong>texto</strong>, um número como 123456 será escrito no arquivo como uma sequência de <i>caracteres</i>: '1', '2', '3', '4', '5', '6' (6 bytes). Durante o processo de leitura, é necessário converter esses caracteres de volta para o número 123456.</p>

<p>No modo <strong>binário</strong>, ao escrever o número 123456 em um arquivo (como um <code>int</code>), o computador armazena a representação binária direta desse valor (geralmente 4 bytes, no formato <i>little-endian</i>). Essa representação é mais compacta, mas não é legível em um editor de texto comum.</p>

<h3>Escrevendo em Arquivos Binários</h3>

<p>Para escrever em um arquivo binário, utilizamos um conjunto de funções focado na manipulação de bytes:</p>

<ul>
    <li><strong style="color: #E0901B;"><code>fopen("arquivo", "wb")</code></strong>: Utilização do modo <strong>"wb"</strong> (Write Binary). Isso cria um novo arquivo para escrita (ou apaga o conteúdo de um arquivo existente): 
        <ul> 
            <li>Use <strong>"ab"</strong> (Append Binary) se quiser adicionar dados ao final de um arquivo existente sem apagá-lo.</li> 
        </ul> 
    </li> 
    <li><strong style="color: #E0901B;"><code>fwrite(buffer, tamanho_item, num_itens, arquivo)</code></strong>: Função capaz de escrever uma quantidade exata de bytes de uma variável para um arquivo. 
        <ul> 
            <li><code>buffer</code>: Ponteiro para a variável que contém os dados que você quer salvar. Por exemplo, <code>&var1</code>.</li> <li><code>tamanho_item</code>: O tamanho, em bytes, de cada item que queremos escrever. Por exemplo, <code>sizeof(int)</code>.</li> 
            <li><code>num_itens</code>: Quantos itens desse tamanho serão escritos (normalmente 1).</li> 
            <li><code>arquivo</code>: O ponteiro para o arquivo aberto.</li> 
        </ul> 
    </li> 
</ul>

<h3>Lendo Arquivos Binários</h3>

<p>Para ler um arquivo binário, utilizamos um conjunto de funções voltadas para a manipulação de bytes:</p>

<ul>
    <li><strong style="color: #E0901B;"><code>fopen("arquivo", "rb")</code></strong>: Utilização do modo <strong>"rb"</strong> (Read Binary). O arquivo é lido como uma sequência de bytes.</li>
    <li><strong style="color: #E0901B;"><code>fread(buffer, tamanho_item, num_itens, arquivo)</code></strong>: Função capaz de ler uma quantidade exata de bytes.
        <ul>
            <li><code>buffer</code>: Ponteiro para a variável onde os dados lidos serão armazenados. Por exemplo, <code>&var1</code>.</li>
            <li><code>tamanho_item</code>: O tamanho, em bytes, de cada item que queremos ler. Por exemplo, <code>sizeof(int)</code>.</li>
            <li><code>num_itens</code>: Quantos itens desse tamanho serão lidos (normalmente 1).</li>
            <li><code>arquivo</code>: O ponteiro para o arquivo aberto.</li>
        </ul>
    </li>
    <li><strong style="color: #E0901B;"><code>fseek(arquivo, offset, origem)</code></strong>: Move o <strong>cursor</strong> de leitura do arquivo:
        <ul>
            <li><code>offset</code>: Quantos bytes pular.</li>
            <li><code>origem</code>: De onde começar o pulo. <code>SEEK_SET</code> (início do arquivo), <code>SEEK_CUR</code> (posição atual) ou <code>SEEK_END</code> (fim do arquivo).</li>
        </ul>
    </li>
</ul>

<h4 style="color: #2d3436;"><strong>Código 1</strong>: Escrevendo o número 123456 em um arquivo no modo texto.</h4>

In [3]:
%%file tex.c

// Comando de compilação: gcc text.c -o out
// Comando de execução: ./out
// Este programa cria um arquivo chamado 'arq.txt'
// No Linux, para visualizar a estrutura do arquivo no disco, utilize o comando: hd arq.txt

#include <stdio.h>

int main() {

    FILE* arquivo = fopen("arq.txt", "w");
    fprintf(arquivo,"123456");
    fclose(arquivo);
    
    return 0;
}

Writing tex.c


<h4 style="color: #2d3436;"><strong>Código 2</strong>: Escrevendo o número 123456 em um arquivo no modo binário.</h4>

In [2]:
%%file bin.c

// Comando de compilação: gcc bin.c -o out
// Comando de execução: ./out
// Este programa cria um arquivo chamado 'arq.bin'
// No Linux, para visualizar a estrutura do arquivo no disco, utilize o comando: hd arq.bin

#include <stdio.h>

int main() {

    FILE* arquivo = fopen("arq.bin", "wb");
    int num = 123456;
    fwrite(&num, sizeof(int), 1, arquivo);
    fclose(arquivo);
    
    return 0;
}

Overwriting bin.c


<p>Podemos utilizar as funções <code>fread</code> e <code>fwrite</code> para manipular os bytes de uma <code>struct</code>. Isso permite a manipulação de estruturas complexas por meio de arquivos binários. Contudo, é necessário conhecer como os dados estão organizados dentro do arquivo binário. Os programas abaixo utilizam uma estrutura chamada <code>Aluno</code>, que é salva em um arquivo binário e posteriormente recuperada.</p>

<h4 style="color: #2d3436;"><strong>Código 3</strong>: Escrevendo uma estrutura (Struct) em um arquivo binário.</h4>

In [1]:
%%file alunos.c

// Comando de compilação: gcc alunos.c -o out
// Comando de execução: ./out
// Este programa cria um arquivo chamado 'alunos.bin'

#include <stdio.h>
#include <stdlib.h>

// O atributo "__attribute__((packed))" é usado pelo compilador GCC para controlar o alinhamento de memória.
// Ele impede a adição de padding (espaços) entre os campos da estrutura (struct).
// Isso garante que o tamanho da struct em memória seja exatamente o mesmo que esperamos no arquivo.
typedef struct __attribute__((packed)) {
    int matricula;
    char nome[50];
    float nota;
} Aluno;

int main() {
    FILE *arquivo = fopen("alunos.bin", "wb");
    if (arquivo == NULL) {
        perror("Erro ao criar arquivo");
        return 1;
    }

    // Criando 3 alunos
    Aluno aluno1 = {101, "Alice", 8.5f};
    Aluno aluno2 = {102, "Bob", 9.0f};
    Aluno aluno3 = {103, "Charlie", 7.2f};

    // Escreve os 3 alunos sequencialmente no arquivo
    fwrite(&aluno1, sizeof(Aluno), 1, arquivo);
    fwrite(&aluno2, sizeof(Aluno), 1, arquivo);
    fwrite(&aluno3, sizeof(Aluno), 1, arquivo);

    fclose(arquivo);

    return 0;
}

Writing alunos.c


<h4 style="color: #2d3436;"><strong>Código 4</strong>: Lendo uma estrutura (Struct) de um arquivo binário.</h4>

In [ ]:
%%file leitura.c

// Comando de compilação: gcc leitura.c -o out
// Comando de execução: ./out
// Este programa faz a leitura do arquivo criado pelo código anterior ('alunos.bin')

#include <stdio.h>
#include <stdlib.h>

typedef struct __attribute__((packed)) {
    int matricula;
    char nome[50];
    float nota;
} Aluno;

int main() {
    FILE *arquivo = fopen("alunos.bin", "rb");
    if (arquivo == NULL) {
        perror("Erro ao abrir alunos.bin");
        return 1;
    }

    // Leitura de todos os alunos do arquivo
    Aluno aluno;
    int i = 0;

    // O loop continua enquanto fread() conseguir ler 1 item completo
    while(fread(&aluno, sizeof(Aluno), 1, arquivo) == 1) {
        printf("Aluno[%d]:\n", i++);
        printf("Matrícula: %d\n", aluno.matricula);
        printf("Nome: %s\n", aluno.nome);
        printf("Matrícula: %.2f\n", aluno.nota);
        printf("\n");
    }
    
    // Rebobina o arquivo para o início
    // Significado: move o cursor para um deslocamento de 0 bytes a partir do início do arquivo.
    fseek(arquivo, 0, SEEK_SET);

    // Calcula o offset para pular o primeiro aluno (índice 0) - leitura do segundo aluno (índice 1)
    long offset = sizeof(Aluno) * 1;
    
    if(fseek(arquivo, offset, SEEK_SET) != 0) {
        perror("Erro ao posicionar o cursor no arquivo\n");
        fclose(arquivo);
        return 1;
    }

    // Lê apenas o segundo aluno
    if (fread(&aluno, sizeof(Aluno), 1, arquivo) == 1) {
        printf("Leitura do segundo aluno (Aluno[1]):\n");
        printf("Matrícula: %d\n", aluno.matricula);
        printf("Nome: %s\n", aluno.nome);
        printf("Matrícula: %.2f\n", aluno.nota);
    }
    else {
        perror("Erro ao ler o segundo aluno\n");
        fclose(arquivo);
        return 1;
    }
    fclose(arquivo);

    return 0;
}

Overwriting leitura.c


<h3>Argumentos de Linha de Comando (argc e argv)</h3>

<p>Em <strong>C</strong>, os parâmetros <code>argc</code> e <code>argv</code> são usados na função <code>main</code> para receber argumentos passados pela linha de comando quando o programa é executado:</p>

<ul>
    <li><code>int argc</code> (Argument Count): Um inteiro que informa <strong>quantos</strong> argumentos foram passados. Este número é sempre 1 ou mais, pois o nome do próprio programa conta como o primeiro argumento.</li>
    <li><code>char *argv[]</code> (Argument Vector): Um array de strings (ponteiros para <code>char</code>). Cada elemento do array aponta para um argumento.
        <ul>
            <li><code>argv[0]</code> é sempre o nome do programa.</li>
            <li><code>argv[1]</code> é o primeiro argumento.</li>
            <li><code>argv[2]</code> é o segundo argumento, e assim por diante.</li>
        </ul>
    </li>
</ul>

<p>No código abaixo, três parâmetros são passados para a função main, os quais são utilizados para determinar o tipo de operação e os operandos usados para produzir a saída do programa.</p>

<h4 style="color: #2d3436;"><strong>Código 5</strong>: Passando argumentos para a função principal do programa.</h4>

In [6]:
%%file main.c

// Comando de compilação: gcc main.c -o calculadora
// Comando de execução: ./calculadora [operação] [num1] [num2]
// Operações válidas: add, sub, mul, div

#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int main(int argc, char* argv[]) {

    if (argc != 4) {
        perror("Erro: Parâmetros invalidos.\n");
        return 1;
    }
    else {
        printf("Nome do programa: %s\n", argv[0]);
        char* operacao = argv[1];
        double num1 = atof(argv[2]);
        double num2 = atof(argv[3]);
        double resultado;

        if (strcmp(operacao, "add") == 0) {
            resultado = num1 + num2;
        } 
        else if (strcmp(operacao, "sub") == 0) {
            resultado = num1 - num2;
        } 
        else if (strcmp(operacao, "mul") == 0) {
            resultado = num1 * num2;
        } 
        else if (strcmp(operacao, "div") == 0) {
            if (num2 == 0) {
                perror("Erro: Divisão por zero.\n");
                return 1;
            }
            resultado = num1 / num2;
        } 
        else {
            perror("Erro: Operação desconhecida.\n");
            return 1;
        }

        printf("Resultado: %.2f\n", resultado);
    }
    
    return 0;
}

Overwriting main.c


<h3>Exercício</h3>

<p>Altere o <strong>Código 4</strong> (<code>leitura.c</code>) de modo que o índice do aluno a ser lido do arquivo <code>alunos.bin</code> seja passado por parâmetro via <code>argv</code>. Considere o seguinte comando de execução:</p>

<pre>./out [indice]</pre>

<p>Onde <code>[indice]</code> representa um índice válido de um dos 3 alunos do arquivo (0, 1 ou 2).</p> 
<p>Se o índice informado for inválido (por exemplo, um número maior que 2 ou um número negativo), o programa deve exibir <strong>todos</strong> os alunos contidos no arquivo.</p>